
# Climate Daily Data Downloader (1995–Present)

This notebook:

- Reads **Climate IDs from an Excel file** in the same folder
- Uses **official Environment Canada daily CSV data**
- Downloads data **from 1995 to today**
- Automatically handles **multiple provinces & missing years**
- Outputs a **single BI-ready DataFrame**

---

## Requirements
- pandas
- openpyxl
- Python 3.9+



## 1. Imports and Configuration


In [1]:

import pandas as pd
from pathlib import Path
from datetime import datetime



## 2. Read Climate IDs from Excel

The Excel file must contain:
- `Climate ID`
- `province`


In [3]:

csv_path = Path("stationlist_clean.csv")  # adjust name if needed

stations_df = pd.read_csv(csv_path)

# Map province names to codes
province_map = {
    "NEW BRUNSWICK": "NB",
    "NOVA SCOTIA": "NS",
    "PRINCE EDWARD ISLAND": "PE",
    "NEWFOUNDLAND AND LABRADOR": "NL",
    "ONTARIO": "ON",
    "QUEBEC": "QC",
    "MANITOBA": "MB",
    "SASKATCHEWAN": "SK",
    "ALBERTA": "AB",
    "BRITISH COLUMBIA": "BC",
    "YUKON": "YT",
    "NORTHWEST TERRITORIES": "NT",
    "NUNAVUT": "NU",
}

stations_df["province_code"] = stations_df["Province"].map(province_map)
stations_df = stations_df.dropna(subset=["Climate ID", "province_code"])

stations_df["Climate ID"] = stations_df["Climate ID"].astype(str)
print("Stations loaded:", stations_df.shape[0])


Stations loaded: 137



## 3. Download Daily Data (1995–Present)


In [4]:

base_url = "https://dd.weather.gc.ca/today/climate/observations/daily/csv"
start_year = 1995
end_year = datetime.now().year

dfs = []

for _, row in stations_df.iterrows():
    climate_id = row["Climate ID"]
    prov = row["province_code"]

    for year in range(start_year, end_year + 1):
        url = (
            f"{base_url}/{prov}/"
            f"climate_daily_{prov}_{climate_id}_{year}_P1D.csv"
        )
        try:
            df = pd.read_csv(
                url,
                encoding="latin-1",
                dtype={"Climate ID": "string"},
                low_memory=False,
            )
            dfs.append(df)
        except Exception:
            continue
print("Files loaded:", len(dfs))


Files loaded: 2730



## 4. Combine and Clean Data


In [5]:

climate_daily = pd.concat(dfs, ignore_index=True)

climate_daily["Date/Time"] = pd.to_datetime(climate_daily["Date/Time"])
climate_daily["Station Name"] = climate_daily["Station Name"].astype("string")

# Convert wind gust speed if present
if "Spd of Max Gust (km/h)" in climate_daily.columns:
    climate_daily["Spd of Max Gust (km/h)"] = pd.to_numeric(
        climate_daily["Spd of Max Gust (km/h)"], errors="coerce"
    )

climate_daily.info()


<class 'pandas.DataFrame'>
RangeIndex: 997119 entries, 0 to 997118
Data columns (total 31 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Longitude (x)              997119 non-null  float64       
 1   Latitude (y)               997119 non-null  float64       
 2   Station Name               997119 non-null  string        
 3   Climate ID                 997119 non-null  string        
 4   Date/Time                  997119 non-null  datetime64[us]
 5   Year                       997119 non-null  int64         
 6   Month                      997119 non-null  int64         
 7   Day                        997119 non-null  int64         
 8   Data Quality               61511 non-null   object        
 9   Max Temp (°C)              828481 non-null  float64       
 10  Max Temp Flag              32583 non-null   object        
 11  Min Temp (°C)              831117 non-null  float64       
 12 

In [6]:
climate_daily =climate_daily.drop(
    columns=[
        'Month', 'Day',
        'Data Quality',
        'Max Temp Flag', 'Min Temp Flag', 'Mean Temp Flag',
        'Heat Deg Days Flag', 'Cool Deg Days Flag', 'Total Rain Flag',
        'Total Snow Flag', 'Total Precip Flag', 'Snow on Grnd Flag',
       'Spd of Max Gust Flag',  'Dir of Max Gust Flag']
)

In [7]:
climate_daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 997119 entries, 0 to 997118
Data columns (total 17 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Longitude (x)              997119 non-null  float64       
 1   Latitude (y)               997119 non-null  float64       
 2   Station Name               997119 non-null  string        
 3   Climate ID                 997119 non-null  string        
 4   Date/Time                  997119 non-null  datetime64[us]
 5   Year                       997119 non-null  int64         
 6   Max Temp (°C)              828481 non-null  float64       
 7   Min Temp (°C)              831117 non-null  float64       
 8   Mean Temp (°C)             826411 non-null  float64       
 9   Heat Deg Days (°C)         826411 non-null  float64       
 10  Cool Deg Days (°C)         826411 non-null  float64       
 11  Total Rain (mm)            169711 non-null  float64       
 12 

In [10]:
climate_daily.isna().sum()

Longitude (x)                     0
Latitude (y)                      0
Station Name                      0
Climate ID                        0
Date/Time                         0
Year                              0
Max Temp (°C)                168638
Min Temp (°C)                166002
Mean Temp (°C)               170708
Heat Deg Days (°C)           170708
Cool Deg Days (°C)           170708
Total Rain (mm)              827408
Total Snow (cm)              831249
Total Precip (mm)            301637
Snow on Grnd (cm)            686434
Dir of Max Gust (10s deg)    533173
Spd of Max Gust (km/h)       528108
dtype: int64

In [11]:
climate_daily["Date/Time"].max()

Timestamp('2026-12-31 00:00:00')

In [12]:

cutoff_date = pd.Timestamp("2026-04-26")

climate_daily_filtered = climate_daily[
    climate_daily["Date/Time"] <= cutoff_date
]


In [13]:
climate_daily_filtered["Date/Time"].max()

Timestamp('2026-04-26 00:00:00')

In [14]:
climate_daily_filtered.info()

<class 'pandas.DataFrame'>
Index: 964749 entries, 0 to 996869
Data columns (total 17 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Longitude (x)              964749 non-null  float64       
 1   Latitude (y)               964749 non-null  float64       
 2   Station Name               964749 non-null  string        
 3   Climate ID                 964749 non-null  string        
 4   Date/Time                  964749 non-null  datetime64[us]
 5   Year                       964749 non-null  int64         
 6   Max Temp (°C)              828481 non-null  float64       
 7   Min Temp (°C)              831117 non-null  float64       
 8   Mean Temp (°C)             826411 non-null  float64       
 9   Heat Deg Days (°C)         826411 non-null  float64       
 10  Cool Deg Days (°C)         826411 non-null  float64       
 11  Total Rain (mm)            169711 non-null  float64       
 12  Tota


## 5. Optional: Save Output


In [15]:
climate_daily_filtered.to_csv("Atlantic_Climate.csv", index=False)

In [ ]:

# Save for BI tools
#climate_daily.to_parquet("climate_daily_1995_present.parquet")
